# 0.setting : pdf를 png로 변환

In [3]:
# from pathlib import Path
# from typing import List, Optional
# import pypdfium2 as pdfium
# from pdf_to_png import pdf_to_png_main

# sample = '../data/H1B1-417-EE102-451_1_CABLE BLOCK DIAGRAM_241121.pdf'
# output_dir = './outputs_png'

# pdf_to_png_main([
#     sample,            # PDF 경로
#     "-o", output_dir,  # 출력 폴더
#     "--dpi", "300"     # DPI 옵션
# ])

# 1.전처리

In [ ]:
"""
1-1. 도면의 테두리 선 삭제(확인용)

input : img_path(테스트할 단일 이미지 위치)
output : 단일 이미지

"""

import cv2
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt

############ 1. 이미지 불러오기
img_path = './outputs_png/H1B1-417-EE102-451_1_CABLE BLOCK DIAGRAM_241121_page_003.png'

############ 1-1 단계. 결과 저장 위치
output_dir = Path("./output_v1")
output_dir.mkdir(parents=True, exist_ok=True)

img = cv2.imread(img_path)
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

# 2. 선 강조 / 잡음 제거
blur = cv2.GaussianBlur(gray, (5,5), 0)
_, thresh = cv2.threshold(blur, 200, 255, cv2.THRESH_BINARY_INV)

# 3. 컨투어 검출
contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

# 4. 잘린 박스 이미지 저장
box_count = 0
for cnt in contours:
    x, y, w, h = cv2.boundingRect(cnt)
    if w > 20 and h > 20:
        # 안쪽 영역 비율 (예: 10%씩 줄이기)
        margin_w = int(w * 0.01)
        margin_h = int(h * 0.01)

        x1 = x + margin_w
        y1 = y + margin_h
        x2 = x + w - margin_w
        y2 = y + h - margin_h

        # 이미지 경계 검사
        x1 = max(x1, 0)
        y1 = max(y1, 0)
        x2 = min(x2, img.shape[1])
        y2 = min(y2, img.shape[0])

        roi = img[y1:y2, x1:x2]
        save_path = output_dir / f"box_{box_count:03d}.png"         # 저장 이미지 수정
        cv2.imwrite(str(save_path), roi)
        box_count += 1

# print(f"총 {box_count}개의 박스 영역 이미지 저장 완료: {output_dir}")


In [ ]:
"""
직접 table 위치(좌표) 확인함
"""
# # Notebook용 인터랙티브 드래그 GUI
# %matplotlib widget

# import matplotlib.pyplot as plt
# from matplotlib.patches import Rectangle
# import cv2
# import numpy as np

# # 이미지 읽기
# img = cv2.imread('./output_boxes_inside/box_000.png')
# img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
# img_copy = img.copy()

# # 좌표 저장
# coords = []

# # Figure, Axis 생성
# fig, ax = plt.subplots(figsize=(8,6))
# ax.imshow(img_rgb)
# ax.set_title("Drag to select a rectangle")

# # 드래그 시작점과 Rectangle 객체
# start_point = [0,0]
# rect_patch = None

# # 마우스 이벤트 콜백
# def on_press(event):
#     global start_point, rect_patch
#     if event.inaxes != ax:
#         return
#     start_point[0], start_point[1] = int(event.xdata), int(event.ydata)
#     if rect_patch:
#         rect_patch.remove()
#         rect_patch = None
#     fig.canvas.draw()

# def on_release(event):
#     global rect_patch
#     if event.inaxes != ax:
#         return
#     x1, y1 = start_point
#     x2, y2 = int(event.xdata), int(event.ydata)
#     coords.append((x1, y1, x2, y2))
    
#     # 좌표를 파일에도 기록
#     with open("selected_coords.txt", "a") as f:
#         f.write(f"{x1},{y1},{x2},{y2}\n")
    
#     print("Selected coordinates:", (x1, y1, x2, y2))
    
#     # 선택 영역 흰색으로 채우기
#     img_copy[min(y1,y2):max(y1,y2), min(x1,x2):max(x1,x2)] = 255
#     ax.imshow(img_copy)
    
#     # 시각화용 Rectangle 추가
#     rect_patch = Rectangle((min(x1,x2), min(y1,y2)), abs(x2-x1), abs(y2-y1),
#                            edgecolor='red', facecolor='none', lw=2)
#     ax.add_patch(rect_patch)
#     fig.canvas.draw()

# # 이벤트 연결
# fig.canvas.mpl_connect('button_press_event', on_press)
# fig.canvas.mpl_connect('button_release_event', on_release)


In [ ]:

"""
1-3. 도면의 table 삭제

좌표 리스트는 이전 단계에서 직접 확인함(고정값)

output : 도면의 표가 삭제된 이미지
"""
import cv2
import numpy as np

img = cv2.imread('./output_v1/box_000.png')

############ 결과 저장 위치
output_dir = Path("./output_v2")
output_dir.mkdir(parents=True, exist_ok=True)

# 좌표 리스트
coords = [
    (4655,3209,3849,2690),
    (3857,3202,2900,2780),
    (3834,2675,4663,3209),
    (2546,3217,3834,2908)
]
for x1, y1, x2, y2 in coords:
    img[min(y1,y2):max(y1,y2), min(x1,x2):max(x1,x2)] = 255

cv2.imwrite('./output_v2/output_remove_table.png', img)


True

In [ ]:
"""
1-4. 도면의 텍스트 삭제

output : 도면의 표가 삭제된 이미지
"""

from drawing_ocr_easy import ocr_main

sample_pdf ='./output_v2/output_remove_table.png'
output_json = './output_v3_text/ocr_results.json'
viz_dir = './output_v2/removed_text'

# argv 형태로 전달 (CLI처럼)
ocr_main([
    sample_pdf,
    '-o', output_json,        # JSON 결과 저장 경로
    '--visualize',            # 시각화 활성화
    '--viz-output', viz_dir   # 시각화 이미지 저장 경로
])

결과 저장 완료: ./output_v3_text/ocr_results.json

OCR 처리 결과 요약
총 이미지 수: 1, 성공: 1, 실패: 0
총 텍스트 라인 수: 105
텍스트 제거 결과 저장: output_v2/preprocessing/output_remove_table.png

첫 번째 이미지 텍스트 샘플:
------------------------------------------------------------
NO.1
SINTER
MAIN WZ14
2F
143
C9
33@071 WZ14
LOCAL
AREA
HIBI-714-SLO2-001
P34I7l1c-Hi8}H}HzLs
417PLC-101C
417LV-1C
143 C9 384J1
0.6/1KV F-CV 1C 240SQ-12L
PLC PANEL
440v INCOMING
ACB PANEL (A:)
ACB PANEL
C-4IZLVIC-_A1
0.6/1Kv F-CW 10C
1.5SQ
C_4I7LVIC-A2
no
0.6/1Kv F-cw
4C 1.5SQ
417DIST-1C
440v DP PANEL
417UPS-1C
UPS PANEL
C_417DISTIC_A1
P_417UPSIC-A1
0.6/1KvV F-CW 7C 1.5SQ 
0.6/1KV F-CV 3C 6SQ
C-41ZUPSIC_A1
0.6/1KV F-CW 7C 1.5SQ
P_417MCCIC-LI,L2,L3
417mCC-1C
144
33EP 4?14
0.6T1Kv F-CV 1C  150SQ-3L
...
